# Chapter 3.4: Graph-Based Retrieval

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** PinSage's random-walk-based graph convolution on billion-scale graphs
2. **Implement** GraphSAGE-style neighborhood aggregation from scratch
3. **Explain** how heterogeneous graphs capture user-item-feature relationships
4. **Describe** knowledge graph embedding methods (TransE, RotatE) for retrieval
5. **Build** a neighborhood sampling and aggregation pipeline
6. **Compare** different aggregation functions: mean, LSTM, pooling
7. **Apply** graph-based embeddings for candidate retrieval

## Prerequisites

- Two-tower models and embedding-based retrieval (Chapter 3.1)
- Basic graph theory concepts (nodes, edges, neighbors)
- PyTorch proficiency

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part3/chapter_3.4_graph_retrieval.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/USERNAME/rec_system/raw/main/notebooks/part3/chapter_3.4_graph_retrieval.ipynb)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import Dict, List, Tuple, Set

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cpu')
print(f"PyTorch version: {torch.__version__}")

## 1. Why Graph-Based Retrieval?

Recommendation data naturally forms graphs:
- **User-Item bipartite graph**: Users connect to items they interacted with
- **Item-Item co-occurrence graph**: Items connect if frequently co-consumed
- **Knowledge graphs**: Items link to attributes (genre, brand, category)

Graph-based methods can capture **higher-order relationships** that simple embedding models miss:
- If user A likes items {X, Y} and user B likes items {Y, Z}, then X and Z are related through the graph path X ← A → Y ← B → Z.

> **💡 Concept:** Graph neural networks propagate information through the graph structure. An item's embedding encodes not just its own features, but the features of items it's connected to — a powerful form of collaborative filtering.

In [ ]:
# Generate a synthetic user-item interaction graph
NUM_USERS = 500
NUM_ITEMS = 2000
NUM_CATEGORIES = 10
NUM_INTERACTIONS = 15000

# Assign items to categories
item_categories = np.random.randint(0, NUM_CATEGORIES, NUM_ITEMS)

# Users have preferences for 2-3 categories
user_prefs = {}
for u in range(NUM_USERS):
    n_prefs = np.random.choice([2, 3])
    user_prefs[u] = np.random.choice(NUM_CATEGORIES, n_prefs, replace=False)

# Generate interactions
adjacency_user_item = defaultdict(set)  # user -> set of items
adjacency_item_user = defaultdict(set)  # item -> set of users
adjacency_item_item = defaultdict(set)  # item -> set of co-interacted items

for _ in range(NUM_INTERACTIONS):
    user = np.random.randint(0, NUM_USERS)
    # 80% from preferred categories
    if np.random.random() < 0.8:
        pref_cat = np.random.choice(user_prefs[user])
        items_in_cat = np.where(item_categories == pref_cat)[0]
        item = np.random.choice(items_in_cat)
    else:
        item = np.random.randint(0, NUM_ITEMS)
    adjacency_user_item[user].add(item)
    adjacency_item_user[item].add(user)

# Build item-item graph from co-interactions
for user, items in adjacency_user_item.items():
    items_list = list(items)
    for i in range(len(items_list)):
        for j in range(i + 1, min(i + 5, len(items_list))):  # limit connections
            adjacency_item_item[items_list[i]].add(items_list[j])
            adjacency_item_item[items_list[j]].add(items_list[i])

avg_user_degree = np.mean([len(v) for v in adjacency_user_item.values()])
avg_item_degree = np.mean([len(v) for v in adjacency_item_item.values() if len(v) > 0])

print(f"Users: {NUM_USERS}, Items: {NUM_ITEMS}")
print(f"Avg items per user: {avg_user_degree:.1f}")
print(f"Avg item-item neighbors: {avg_item_degree:.1f}")
print(f"Items with neighbors: {len(adjacency_item_item)}")

## 2. GraphSAGE: Inductive Node Embedding

GraphSAGE (Hamilton et al., 2017) learns embeddings by **sampling and aggregating** neighbor features. Unlike transductive methods (DeepWalk, Node2Vec), GraphSAGE is inductive — it can embed new nodes.

### Algorithm

For each layer $l$:

$$\mathbf{h}^{(l)}_v = \sigma\left(\mathbf{W}^{(l)} \cdot \text{CONCAT}\left(\mathbf{h}^{(l-1)}_v, \text{AGG}\left(\{\mathbf{h}^{(l-1)}_u : u \in \mathcal{N}(v)\}\right)\right)\right)$$

where $\mathcal{N}(v)$ is a sampled subset of $v$'s neighbors.

### Aggregation Functions
1. **Mean**: $\text{AGG} = \frac{1}{|\mathcal{N}|} \sum_{u \in \mathcal{N}} \mathbf{h}_u$
2. **Max-Pool**: $\text{AGG} = \max(\{\sigma(\mathbf{W}_{\text{pool}} \mathbf{h}_u + \mathbf{b})\})$
3. **LSTM**: Apply LSTM to a random permutation of neighbors

> **⚠️ Common Pitfall:** Neighbor sampling is critical for scalability. Without it, the computation graph explodes exponentially with depth. PinSage uses importance-based sampling instead of uniform sampling.

In [ ]:
class NeighborSampler:
    """Sample fixed-size neighborhoods for graph neural networks."""
    
    def __init__(self, adjacency: Dict[int, Set[int]]):
        self.adjacency = adjacency
    
    def sample(self, nodes: List[int], num_samples: int) -> Tuple[List[List[int]], List[int]]:
        """Sample neighbors for a batch of nodes.
        
        Args:
            nodes: list of node IDs
            num_samples: number of neighbors to sample per node
        
        Returns:
            sampled_neighbors: list of sampled neighbor lists
            all_unique_nodes: all unique nodes needed for computation
        """
        sampled = []
        all_nodes = set(nodes)
        
        for node in nodes:
            neighbors = list(self.adjacency.get(node, set()))
            if len(neighbors) == 0:
                # Self-loop for isolated nodes
                sampled.append([node] * num_samples)
            elif len(neighbors) >= num_samples:
                chosen = list(np.random.choice(neighbors, num_samples, replace=False))
                sampled.append(chosen)
            else:
                # Sample with replacement if not enough neighbors
                chosen = list(np.random.choice(neighbors, num_samples, replace=True))
                sampled.append(chosen)
            all_nodes.update(sampled[-1])
        
        return sampled, list(all_nodes)


class GraphSAGELayer(nn.Module):
    """Single GraphSAGE layer with mean aggregation."""
    
    def __init__(self, input_dim: int, output_dim: int, aggregator: str = 'mean'):
        super().__init__()
        self.aggregator = aggregator
        
        if aggregator == 'mean':
            self.W = nn.Linear(input_dim * 2, output_dim)
        elif aggregator == 'pool':
            self.pool_layer = nn.Linear(input_dim, input_dim)
            self.W = nn.Linear(input_dim * 2, output_dim)
        else:
            raise ValueError(f"Unknown aggregator: {aggregator}")
    
    def forward(self, node_features: torch.Tensor, 
                neighbor_features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            node_features: (B, D) features of target nodes
            neighbor_features: (B, K, D) features of sampled neighbors
        """
        if self.aggregator == 'mean':
            agg = neighbor_features.mean(dim=1)  # (B, D)
        elif self.aggregator == 'pool':
            agg = F.relu(self.pool_layer(neighbor_features))  # (B, K, D)
            agg = agg.max(dim=1).values  # (B, D)
        
        combined = torch.cat([node_features, agg], dim=-1)  # (B, 2D)
        output = F.relu(self.W(combined))  # (B, D_out)
        return F.normalize(output, p=2, dim=-1)


class GraphSAGE(nn.Module):
    """Multi-layer GraphSAGE model."""
    
    def __init__(self, num_nodes: int, input_dim: int, hidden_dim: int,
                 output_dim: int, num_layers: int = 2, aggregator: str = 'mean'):
        super().__init__()
        self.num_layers = num_layers
        self.node_embedding = nn.Embedding(num_nodes, input_dim)
        
        self.layers = nn.ModuleList()
        dims = [input_dim] + [hidden_dim] * (num_layers - 1) + [output_dim]
        for i in range(num_layers):
            self.layers.append(GraphSAGELayer(dims[i], dims[i+1], aggregator))
    
    def forward(self, node_ids: torch.Tensor, 
                neighbor_ids_per_layer: List[torch.Tensor]) -> torch.Tensor:
        """
        Args:
            node_ids: (B,) target node IDs
            neighbor_ids_per_layer: list of (B, K) neighbor ID tensors per layer
        """
        h = self.node_embedding(node_ids)  # (B, D)
        
        for layer_idx, layer in enumerate(self.layers):
            neighbor_ids = neighbor_ids_per_layer[layer_idx]  # (B, K)
            neighbor_h = self.node_embedding(neighbor_ids)  # (B, K, D)
            h = layer(h, neighbor_h)
        
        return h


# Test GraphSAGE
sage = GraphSAGE(NUM_ITEMS, input_dim=32, hidden_dim=32, output_dim=32, num_layers=2)
print(f"GraphSAGE parameters: {sum(p.numel() for p in sage.parameters()):,}")

# Sample some nodes and their neighbors
sampler = NeighborSampler(adjacency_item_item)
test_nodes = list(range(16))
neighbors_l1, _ = sampler.sample(test_nodes, num_samples=10)
neighbors_l2, _ = sampler.sample(test_nodes, num_samples=10)

node_ids = torch.tensor(test_nodes)
neigh_l1 = torch.tensor(neighbors_l1)
neigh_l2 = torch.tensor(neighbors_l2)

embeddings = sage(node_ids, [neigh_l1, neigh_l2])
print(f"Output embeddings shape: {embeddings.shape}")

## 3. PinSage (Pinterest, 2018)

PinSage (Ying et al., 2018, Pinterest) is a production-grade GraphSAGE variant for billion-scale recommendation. Key innovations:

1. **Importance-based sampling**: Instead of uniform sampling, use random walk-based importance scores. Neighbors visited more often during random walks from a node are more important.

2. **Hard negative mining**: Use the current model to find hard negatives from the graph.

3. **Curriculum training**: Start with easy negatives, gradually introduce harder ones.

4. **Producer-consumer architecture**: Separate model computation from graph traversal for efficiency.

### Importance-based Neighbor Selection

For node $v$, perform $T$ random walks of length $L$. The importance of neighbor $u$ is:

$$w(u | v) = \frac{\text{count}(u \text{ visited from } v)}{T}$$

Select top-$K$ neighbors by importance, then normalize:

$$\alpha(u | v) = \frac{w(u | v)}{\sum_{u' \in \mathcal{N}_K(v)} w(u' | v)}$$

In [ ]:
class PinSageNeighborSampler:
    """PinSage-style importance-based neighbor sampling via random walks."""
    
    def __init__(self, adjacency: Dict[int, Set[int]], 
                 num_walks: int = 100, walk_length: int = 5):
        self.adjacency = adjacency
        self.num_walks = num_walks
        self.walk_length = walk_length
    
    def compute_importance(self, node: int) -> Dict[int, float]:
        """Compute neighbor importance via random walks."""
        visit_counts = defaultdict(int)
        neighbors = list(self.adjacency.get(node, set()))
        
        if not neighbors:
            return {node: 1.0}
        
        for _ in range(self.num_walks):
            current = node
            for _ in range(self.walk_length):
                nbrs = list(self.adjacency.get(current, set()))
                if not nbrs:
                    break
                current = np.random.choice(nbrs)
                if current != node:
                    visit_counts[current] += 1
        
        # Normalize
        total = sum(visit_counts.values()) or 1
        return {k: v / total for k, v in visit_counts.items()}
    
    def sample(self, nodes: List[int], num_samples: int) -> Tuple[List[List[int]], List[List[float]]]:
        """Sample neighbors weighted by importance."""
        all_neighbors = []
        all_weights = []
        
        for node in nodes:
            importance = self.compute_importance(node)
            
            if not importance:
                all_neighbors.append([node] * num_samples)
                all_weights.append([1.0 / num_samples] * num_samples)
                continue
            
            # Sort by importance and take top-K
            sorted_neighbors = sorted(importance.items(), key=lambda x: -x[1])
            top_k = sorted_neighbors[:num_samples]
            
            if len(top_k) < num_samples:
                # Pad with repetitions
                while len(top_k) < num_samples:
                    top_k.append(top_k[np.random.randint(len(top_k))])
            
            nbrs = [n for n, _ in top_k]
            weights = [w for _, w in top_k]
            total_w = sum(weights)
            weights = [w / total_w for w in weights]
            
            all_neighbors.append(nbrs)
            all_weights.append(weights)
        
        return all_neighbors, all_weights


# Compare uniform vs importance sampling
pinsage_sampler = PinSageNeighborSampler(adjacency_item_item, num_walks=50, walk_length=4)
uniform_sampler = NeighborSampler(adjacency_item_item)

test_node = 0
importance = pinsage_sampler.compute_importance(test_node)

# Visualize importance distribution
if importance:
    sorted_imp = sorted(importance.values(), reverse=True)[:30]
    plt.figure(figsize=(10, 4))
    plt.bar(range(len(sorted_imp)), sorted_imp, color='steelblue', alpha=0.8)
    plt.xlabel('Neighbor Rank', fontsize=12)
    plt.ylabel('Importance Score', fontsize=12)
    plt.title(f'PinSage Importance Distribution for Node {test_node}', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"Node {test_node} has {len(importance)} reachable neighbors")
    print(f"Top 5 by importance: {sorted(importance.items(), key=lambda x: -x[1])[:5]}")

## 4. Heterogeneous Graph Retrieval

Real recommendation systems have heterogeneous graphs with multiple node and edge types:

- **Node types**: user, item, category, brand, tag
- **Edge types**: click, purchase, belongs-to-category, has-brand

Heterogeneous graph methods use different transformation matrices for different relation types:

$$\mathbf{h}_v^{(l)} = \sigma\left(\sum_{r \in \mathcal{R}} \sum_{u \in \mathcal{N}_r(v)} \frac{1}{|\mathcal{N}_r(v)|} \mathbf{W}_r^{(l)} \mathbf{h}_u^{(l-1)}\right)$$

> **🔑 Pro Tip:** In practice, heterogeneous graph models provide a significant boost over homogeneous ones because they can capture the semantics of different relationship types. For example, "purchased" and "browsed" edges should have different weights.

In [ ]:
class HeterogeneousGraphSAGE(nn.Module):
    """GraphSAGE for heterogeneous graphs with typed edges."""
    
    def __init__(self, node_types: Dict[str, int], embed_dim: int = 32,
                 hidden_dim: int = 32, edge_types: List[str] = None):
        super().__init__()
        self.embed_dim = embed_dim
        
        # Per-type node embeddings
        self.node_embeddings = nn.ModuleDict({
            ntype: nn.Embedding(count, embed_dim)
            for ntype, count in node_types.items()
        })
        
        # Per-edge-type transformation
        if edge_types is None:
            edge_types = ['interact', 'belong_to']
        self.edge_transforms = nn.ModuleDict({
            etype: nn.Linear(embed_dim, hidden_dim)
            for etype in edge_types
        })
        
        # Combine self + neighbors
        self.combine = nn.Linear(embed_dim + hidden_dim, embed_dim)
    
    def forward(self, target_type: str, target_ids: torch.Tensor,
                neighbor_info: Dict[str, Tuple[str, torch.Tensor]]):
        """
        Args:
            target_type: type of target nodes
            target_ids: (B,) target node IDs
            neighbor_info: dict of {edge_type: (node_type, neighbor_ids)}
                where neighbor_ids is (B, K)
        """
        self_emb = self.node_embeddings[target_type](target_ids)  # (B, D)
        
        agg_list = []
        for edge_type, (node_type, neighbor_ids) in neighbor_info.items():
            n_emb = self.node_embeddings[node_type](neighbor_ids)  # (B, K, D)
            n_agg = n_emb.mean(dim=1)  # (B, D)
            transformed = self.edge_transforms[edge_type](n_agg)  # (B, H)
            agg_list.append(transformed)
        
        # Average across edge types
        neighbor_agg = torch.stack(agg_list, dim=0).mean(dim=0)  # (B, H)
        
        combined = torch.cat([self_emb, neighbor_agg], dim=-1)  # (B, D+H)
        output = F.relu(self.combine(combined))
        return F.normalize(output, p=2, dim=-1)


# Test heterogeneous GraphSAGE
het_model = HeterogeneousGraphSAGE(
    node_types={'user': NUM_USERS, 'item': NUM_ITEMS, 'category': NUM_CATEGORIES},
    embed_dim=32, hidden_dim=32,
    edge_types=['interact', 'belong_to']
)

B = 16
item_ids = torch.randint(0, NUM_ITEMS, (B,))
user_neighbors = torch.randint(0, NUM_USERS, (B, 5))  # 5 user neighbors per item
cat_neighbors = torch.tensor([item_categories[i.item()] for i in item_ids]).unsqueeze(1).expand(B, 3)

item_emb = het_model(
    'item', item_ids,
    {'interact': ('user', user_neighbors), 'belong_to': ('category', cat_neighbors)}
)
print(f"Heterogeneous item embeddings: {item_emb.shape}")
print(f"Model parameters: {sum(p.numel() for p in het_model.parameters()):,}")

## 5. Knowledge Graph Embeddings: TransE and RotatE

Knowledge graph embeddings model triplets $(h, r, t)$ — head entity, relation, tail entity.

### TransE (Bordes et al., 2013)
$$\mathbf{h} + \mathbf{r} \approx \mathbf{t}$$

Score: $f(h, r, t) = -\|\mathbf{h} + \mathbf{r} - \mathbf{t}\|$

### RotatE (Sun et al., 2019)
Models relations as rotations in complex space:
$$\mathbf{t} = \mathbf{h} \circ \mathbf{r}, \quad |r_i| = 1$$

where $\circ$ is element-wise complex multiplication and $\mathbf{r}$ has unit modulus.

RotatE can model symmetric, antisymmetric, inverse, and compositional relations — unlike TransE which cannot model symmetric relations.

> **💡 Concept:** For recommendation, a knowledge graph might have triplets like (iPhone, is_brand, Apple), (iPhone, has_category, Electronics). KG embeddings enrich item representations with structured knowledge.

In [ ]:
class TransE(nn.Module):
    """TransE knowledge graph embedding (Bordes et al., 2013)."""
    
    def __init__(self, num_entities: int, num_relations: int, embed_dim: int = 32):
        super().__init__()
        self.entity_embedding = nn.Embedding(num_entities, embed_dim)
        self.relation_embedding = nn.Embedding(num_relations, embed_dim)
        
        nn.init.xavier_uniform_(self.entity_embedding.weight)
        nn.init.xavier_uniform_(self.relation_embedding.weight)
    
    def forward(self, head: torch.Tensor, relation: torch.Tensor, 
                tail: torch.Tensor) -> torch.Tensor:
        h = self.entity_embedding(head)
        r = self.relation_embedding(relation)
        t = self.entity_embedding(tail)
        
        # Score: negative L2 distance
        score = -torch.norm(h + r - t, p=2, dim=-1)
        return score
    
    def loss(self, pos_head, pos_rel, pos_tail, neg_tail, margin=1.0):
        pos_score = self.forward(pos_head, pos_rel, pos_tail)
        neg_score = self.forward(pos_head, pos_rel, neg_tail)
        return F.relu(margin + neg_score - pos_score).mean()


class RotatE(nn.Module):
    """RotatE: rotation-based KG embedding (Sun et al., 2019)."""
    
    def __init__(self, num_entities: int, num_relations: int, embed_dim: int = 32):
        super().__init__()
        # Entity embeddings in complex space (real + imaginary)
        self.entity_re = nn.Embedding(num_entities, embed_dim)
        self.entity_im = nn.Embedding(num_entities, embed_dim)
        
        # Relation: phase angles (unit modulus in complex plane)
        self.relation_phase = nn.Embedding(num_relations, embed_dim)
        
        nn.init.xavier_uniform_(self.entity_re.weight)
        nn.init.xavier_uniform_(self.entity_im.weight)
        nn.init.uniform_(self.relation_phase.weight, -np.pi, np.pi)
    
    def forward(self, head, relation, tail):
        h_re = self.entity_re(head)
        h_im = self.entity_im(head)
        t_re = self.entity_re(tail)
        t_im = self.entity_im(tail)
        
        phase = self.relation_phase(relation)
        r_re = torch.cos(phase)
        r_im = torch.sin(phase)
        
        # Complex multiplication: (h_re + i*h_im) * (r_re + i*r_im)
        rot_re = h_re * r_re - h_im * r_im
        rot_im = h_re * r_im + h_im * r_re
        
        # Distance to tail
        score = -torch.sqrt((rot_re - t_re) ** 2 + (rot_im - t_im) ** 2 + 1e-9).sum(dim=-1)
        return score


# Train TransE on synthetic KG
NUM_ENTITIES = NUM_ITEMS + NUM_CATEGORIES
NUM_RELATIONS = 3  # has_category, co_purchased, similar_to

# Generate KG triplets
triplets = []
for item_id in range(NUM_ITEMS):
    cat_id = item_categories[item_id] + NUM_ITEMS  # offset category IDs
    triplets.append((item_id, 0, cat_id))  # has_category

# Add some co-purchase edges
for item, neighbors in list(adjacency_item_item.items())[:500]:
    for neighbor in list(neighbors)[:3]:
        triplets.append((item, 1, neighbor))  # co_purchased

triplets = np.array(triplets)

# Train
transe = TransE(NUM_ENTITIES, NUM_RELATIONS, embed_dim=32)
optimizer = torch.optim.Adam(transe.parameters(), lr=0.01)

losses = []
for epoch in range(50):
    idx = np.random.permutation(len(triplets))[:512]
    batch = triplets[idx]
    
    head = torch.tensor(batch[:, 0], dtype=torch.long)
    rel = torch.tensor(batch[:, 1], dtype=torch.long)
    tail = torch.tensor(batch[:, 2], dtype=torch.long)
    neg_tail = torch.randint(0, NUM_ENTITIES, (len(batch),))
    
    loss = transe.loss(head, rel, tail, neg_tail)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

plt.figure(figsize=(8, 4))
plt.plot(losses, color='teal', linewidth=1.5, alpha=0.8)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Margin Loss', fontsize=12)
plt.title('TransE Training on Recommendation KG', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. End-to-End Graph Retrieval Pipeline

Let's train a complete GraphSAGE model for item retrieval and evaluate it.

In [ ]:
# Train GraphSAGE for item-item similarity
sage_model = GraphSAGE(NUM_ITEMS, input_dim=32, hidden_dim=32, output_dim=32, 
                       num_layers=2, aggregator='mean')
optimizer = torch.optim.Adam(sage_model.parameters(), lr=1e-3)
sampler = NeighborSampler(adjacency_item_item)

# Training: use co-interacted items as positives
all_edges = []
for item, neighbors in adjacency_item_item.items():
    for neighbor in neighbors:
        all_edges.append((item, neighbor))
all_edges = np.array(all_edges)

train_losses = []
sage_model.train()

for epoch in range(30):
    idx = np.random.permutation(len(all_edges))[:1024]
    batch_edges = all_edges[idx]
    
    src_nodes = batch_edges[:, 0].tolist()
    pos_nodes = batch_edges[:, 1].tolist()
    
    # Sample neighbors for source and positive nodes
    src_n1, _ = sampler.sample(src_nodes, 5)
    src_n2, _ = sampler.sample(src_nodes, 5)
    pos_n1, _ = sampler.sample(pos_nodes, 5)
    pos_n2, _ = sampler.sample(pos_nodes, 5)
    
    src_ids = torch.tensor(src_nodes)
    pos_ids = torch.tensor(pos_nodes)
    
    src_emb = sage_model(src_ids, [torch.tensor(src_n1), torch.tensor(src_n2)])
    pos_emb = sage_model(pos_ids, [torch.tensor(pos_n1), torch.tensor(pos_n2)])
    
    # In-batch contrastive loss
    logits = torch.matmul(src_emb, pos_emb.T) * 20.0
    labels = torch.arange(logits.size(0))
    loss = F.cross_entropy(logits, labels)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:2d} | Loss: {loss.item():.4f}")

plt.figure(figsize=(8, 4))
plt.plot(train_losses, 'o-', color='purple', linewidth=2, markersize=3)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Contrastive Loss', fontsize=12)
plt.title('GraphSAGE Training for Item Retrieval', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercises

### 🏋️ Exercise 1: Implement Max-Pool Aggregation

GraphSAGE supports different aggregation functions. Implement and compare mean vs. max-pool aggregation.

In [ ]:
class PoolAggregator(nn.Module):
    """Max-pool aggregator for GraphSAGE."""
    def __init__(self, input_dim, output_dim):
        super().__init__()
        # TODO: Define a learnable transformation before pooling
        # TODO: Define the combining layer
        pass
    
    def forward(self, node_features, neighbor_features):
        # TODO: Transform neighbor features
        # TODO: Element-wise max pool
        # TODO: Concatenate with self features and project
        pass

# Compare mean vs pool aggregation on the same data
# TODO: Train two GraphSAGE models with different aggregators
# TODO: Compare training curves and retrieval quality

### 🏋️ Exercise 2: Implement PinSage with Curriculum Hard Negatives

Build the PinSage training pipeline with progressively harder negatives.

In [ ]:
def pinsage_curriculum_training(model, adjacency, all_edges, 
                                 num_epochs=30, hard_neg_start_epoch=10):
    """
    Train PinSage-style model with curriculum learning.
    
    Phases:
    1. Epochs 0 to hard_neg_start_epoch: random negatives only
    2. Epochs hard_neg_start_epoch+: mix of random + hard negatives
       (mined from current model's nearest neighbors)
    
    Returns:
        Training losses per epoch
    """
    # TODO: Implement phase 1 with random negatives
    # TODO: At hard_neg_start_epoch, compute all item embeddings
    # TODO: For each positive pair, find hard negatives (close but not connected)
    # TODO: Continue training with mixed negatives
    pass

### 🏋️ Exercise 3: Build a Full Heterogeneous Graph Retrieval System

Combine user, item, and category nodes in a heterogeneous graph. Train the model and evaluate retrieval.

In [ ]:
def train_heterogeneous_retrieval(num_users, num_items, num_categories,
                                   user_item_edges, item_categories,
                                   embed_dim=32, num_epochs=20):
    """
    Train a heterogeneous graph model for retrieval.
    
    Steps:
    1. Build heterogeneous graph with user, item, category nodes
    2. Define edge types: user-clicks-item, item-belongs-to-category
    3. Train with link prediction objective
    4. Evaluate: for each user, retrieve items using graph embeddings
    
    Returns:
        Trained model, item embeddings, recall metrics
    """
    # TODO: Create HeterogeneousGraphSAGE model
    # TODO: Sample training batches
    # TODO: Train with contrastive loss
    # TODO: Evaluate Recall@K
    pass

## Summary

| Method | Key Idea | Scale | Pros | Cons |
|--------|----------|-------|------|------|
| **GraphSAGE** | Sample & aggregate neighbors | Millions | Inductive, flexible | Uniform sampling suboptimal |
| **PinSage** | Importance sampling + curriculum | Billions | Production-proven | Complex training pipeline |
| **HetGNN** | Typed edges + per-relation transforms | Millions | Rich semantics | More parameters |
| **TransE** | Translation in embedding space | Millions | Simple, effective | Can't model symmetric relations |
| **RotatE** | Rotation in complex space | Millions | Models all relation types | Higher compute cost |

**Next up**: Chapter 3.5 covers Sequential Retrieval — using the temporal order of interactions for better retrieval.